# Setup

## Load packages

In [19]:
import folium
import pandas as pd
import geopandas as gpd
import os
from folium import Map, Marker, FeatureGroup, LayerControl
from folium.features import CustomIcon
import branca
from folium.plugins import LocateControl
from folium import Element
import yaml
from pyprojroot import here
from jinja2 import Template
from pathlib import Path

## Read yaml settings

In [40]:
with open(here("config.yaml", "r")) as f:
    config = yaml.safe_load(f)


title = config['title']
description = config['description']
tier_dict = config['tier_dict']
icon_file_type = config['icon_file_type']
basemap_type = config['basemap_type']

# Load data

In [5]:
# reading data
location_df = pd.read_csv(here('data/locations.csv'))

# Coverting to geodataframe
locations_geometry = gpd.points_from_xy(location_df.longitude, location_df.latitude)
locations_gdf = gpd.GeoDataFrame(location_df, crs='EPSG:4326', geometry=locations_geometry)

# Mapping

## Style popups

In [6]:
# Define popup content
popup_template = """
<div style="
    display: flex; 
    border: 2px solid grey; 
    border-radius: 10px; 
    padding: 10px; 
    width: 290px;  /* slightly wider overall */
    min-height: 120px; 
    box-shadow: 3px 3px 8px rgba(0,0,0,0.4);
">
    <div style="
        flex: 2.2; 
        margin-right: 10px; 
        border-right: 2px solid grey; 
        padding-right: 10px;
    ">
        <h4 style="margin: 0;">{location}</h4>
        <p style="margin: 5px 0;">Standout Quality: {description}</p>
    </div>
    <div style="
        flex: 1.2;  /* slightly wider image column */
        border-left: 2px solid grey; 
        padding-left: 10px; 
        display: flex; 
        align-items: flex-start;
    ">
        <img src="{image}" alt="Image" style="
            width: 70px;  /* slightly wider image */
            height: 100px; 
            object-fit: cover; 
            border-radius: 5px;
        ">
    </div>
</div>
"""

# Define a dictionary of icon paths for each tier
icon_paths = {tier: str(here(f"docs/icons/tier_{tier}.{icon_file_type}")) for tier in range(1, len(tier_dict) + 1)}

## Generating the map

In [7]:
# Create the map without specifying a location
m = folium.Map(
    zoom_start=12,  # Initial zoom level
    tiles=basemap_type
)

# Use tiers for layers
tier_order = list(range(1, len(tier_dict) + 1))

# Initialize a list to store all coordinates
all_coordinates = []

# Loop through each tier
for tier_value in tier_order:
    fg = folium.FeatureGroup(name=f"Tier {tier_value}")
    tier_gdf = locations_gdf[locations_gdf["tier"] == tier_value]

    # Dynamically set the icon path based on the tier
    icon_path = icon_paths.get(tier_value)

    for _, row in tier_gdf.iterrows():
        icon = folium.CustomIcon(
            icon_image=icon_path,
            icon_size=(30, 30),
            icon_anchor=(20, 20)
        )

        popup_html = popup_template.format(
            location=row["location"],
            description=row["description"] if pd.notna(row["description"]) else "",  # Default description
            image=f"icons/popups/{row['location']}.png" if pd.notna(row["location"]) else "/icons/popups/default_image.png",  # Dynamically source image
            lat=round(row.geometry.y, 5),
            lon=round(row.geometry.x, 5)
        )

        # Add marker to the map
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            icon=icon,
            tooltip=row["location"],
            popup=popup_html
        ).add_to(fg)

        # Append the coordinates to the list
        all_coordinates.append([row.geometry.y, row.geometry.x])

    fg.add_to(m)

# Automatically adjust the map to fit all markers with padding
if all_coordinates:
    # Calculate the bounds
    min_lat = min(coord[0] for coord in all_coordinates)
    max_lat = max(coord[0] for coord in all_coordinates)
    min_lon = min(coord[1] for coord in all_coordinates)
    max_lon = max(coord[1] for coord in all_coordinates)

    # Add padding to the bounds (e.g., 0.01 degrees for both latitude and longitude)
    padding = 0.01
    padded_bounds = [[min_lat - padding, min_lon - padding], [max_lat + padding, max_lon + padding]]

    # Fit the map to the padded bounds
    m.fit_bounds(padded_bounds)

# Add layer control and locate control
folium.LayerControl().add_to(m)
LocateControl().add_to(m)

## Styling the legend

In [53]:
# Dynamically generate the legend HTML
legend_items = ""
for tier in range(1, len(tier_dict) + 1):
    phrase = tier_dict.get(tier, f"Tier {tier}")  # Default to "Tier {tier}" if no phrase is found
    legend_items += f"""
        <li style="margin-bottom: 10px;">  <!-- Added spacing between items -->
            <img src="icons/tier_{tier}.{icon_file_type}" style="height: 62px; vertical-align: middle;">  <!-- Adjusted dynamically -->
            – {phrase}
        </li>
    """

legend_html = f"""
<div style="
    position: fixed; 
    bottom: 20px;  /* Decreased bottom margin */
    left: 20px;  /* Decreased left margin */
    width: 320px;  /* Slightly increased width */
    z-index:9999; 
    font-size:16px;  /* Font size remains the same */
    background-color: white;
    border:3px solid grey;  /* Thicker border */
    border-radius:10px;  /* Slightly larger border radius */
    padding: 15px;  /* Padding remains the same */
    box-shadow: 3px 3px 8px rgba(0,0,0,0.4);  /* Slightly larger shadow */
">
    <div style="text-align: left; margin-bottom: 15px;">
        <br><strong style="font-size:18px;">The Cheesesteak Olympics</strong>  <!-- Larger title -->
    </div>
    <ul style="list-style: none; padding-left: 10px;">
        {legend_items}
    </ul>
</div>
"""

m.get_root().html.add_child(Element(legend_html))

# Saving map as html doc

In [35]:
# save the map
m.save(here("docs/map.html"))

# Creating html template with ribbon

## Generate dynamic legend items

In [36]:
# Generate dynamic legend items
legend_items = ""
for tier in range(1, len(tier_dict) + 1):
    phrase = tier_dict.get(tier, f"Tier {tier}")
    legend_items += f"""
        <li style="margin-bottom: 10px;">
            <img src="icons/tier_{tier}.{icon_file_type}" style="height: 32px; vertical-align: middle;">
            – {phrase}
        </li>
    """

## Injecting folium map into template

In [41]:
# Load your template
template_path = here("templates/template_with_sidebar.html")
with open(template_path) as f:
    template = Template(f.read())

# Render final HTML with legend
html_out = template.render(
    legend_items=legend_items,
    title=config["title"],
    description=config["description"]
)

# Save to docs/index.html
here("docs/index.html").write_text(html_out, encoding="utf-8")


2245